# Semana 9: Práctica. Betas, alfas y factores con datos reales

**Curso:** Tópicos de Finanzas Avanzadas (ECON-421, UPAO 2026-20)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JonathanRosasV/topicos-finanzas-upao/blob/main/09_capm_multifactor/clase09_practica.ipynb)

Hoy ponemos a prueba la teoría con diez años de datos: estimamos el CAPM para nuestras 14 acciones, dibujamos la SML empírica, bajamos los factores de la biblioteca de Kenneth French, corremos regresiones de tres factores y terminamos leyendo el estilo de un fondo solo con sus cargas.

**Requisito previo:** `git pull` en tu fork. Esta semana se agregan `utils/french.py` (descarga de factores) y, en `utils/finanzas.py`, `alfa_capm()`, `descomponer_riesgo()`, `retorno_multifactor()` y `regresion_factores()`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import statsmodels.api as sm

from utils.french import get_factores
from utils.finanzas import regresion_factores

ACCIONES = ["ALICORC1.LM", "FERREYC1.LM", "CPACASC1.LM", "LUSURC1.LM",      # BVL (en soles)
            "BAP", "BVN", "SCCO", "IFS",                                    # Peru en NYSE
            "AAPL", "MSFT", "JNJ", "KO", "XOM", "JPM"]                      # Estados Unidos
FONDOS = {"SPY": "S&P 500", "IWM": "Russell 2000 (pequenas)", "IWN": "Russell 2000 valor",
          "VTV": "Vanguard valor", "VUG": "Vanguard crecimiento", "QQQ": "Nasdaq 100",
          "ARKK": "ARK Innovation", "BRK-B": "Berkshire Hathaway", "EPU": "MSCI Peru"}

## 1. Los datos: retornos en exceso y factores

Tres cuidados de consistencia, los mismos de siempre:

- **Moneda.** Todo en dólares: las acciones de la BVL se convierten con el tipo de cambio de cada mes.
- **Retornos en exceso.** El CAPM y Fama-French se estiman sobre $R_i - R_f$. Usamos la tasa libre de riesgo mensual que publica French (letra del Tesoro a 1 mes), en la misma moneda y frecuencia que los retornos.
- **Fechas.** Yahoo etiqueta cada mes con su primer día y French con `AAAAMM`. Llevamos ambos a un índice mensual para que se alineen. French publica con uno o dos meses de rezago, así que la muestra termina donde terminan sus factores.

Usamos 10 años: las regresiones de factores necesitan muestra.

In [ ]:
bruto = yf.download(ACCIONES + list(FONDOS) + ["PEN=X"], period="10y", interval="1mo",
                    auto_adjust=True, progress=False)["Close"].iloc[:-1]
tc = bruto["PEN=X"].ffill()
precios = bruto.drop(columns="PEN=X")
locales = [t for t in precios.columns if t.endswith(".LM")]
precios[locales] = precios[locales].div(tc, axis=0)                      # de soles a dolares

ret = precios.ffill().pct_change(fill_method=None).iloc[1:]
ret.index = ret.index.to_period("M")

ff = get_factores("ff3")                                                 # Mkt-RF, SMB, HML y RF, en decimales
ff = ff.loc[ret.index[0]:]
exceso = ret.sub(ff["RF"], axis=0).loc[:ff.index[-1]]                    # retornos en exceso, alineados

print(f"Muestra: {exceso.index[0]} a {exceso.index[-1]} ({len(exceso)} meses)")
print("\nPrimas anualizadas de los factores en la muestra:")
print((ff.loc[exceso.index].mean() * 12).round(3).to_string())

**Para discutir:** compara las primas de esta muestra con las históricas de la clase (mercado 7.2%, SMB 1.8%, HML 3.6%). Diez años es poco: una prima de factor puede ser negativa durante una década entera. ¿Qué le pasó al tamaño y al valor en estos años?

## 2. El CAPM de cada acción

Una regresión por acción: retorno en exceso contra el exceso del mercado. La pendiente es el beta, el intercepto anualizado es el alfa de Jensen y el $R^2$ es la fracción sistemática del riesgo.

In [ ]:
capm_tabla = pd.DataFrame({t: regresion_factores(exceso[t], ff[["Mkt-RF"]]) for t in ACCIONES}).T
capm_tabla = capm_tabla.rename(columns={"b_Mkt-RF": "beta", "t_Mkt-RF": "t_beta"})
capm_tabla["retorno en exceso"] = exceso[ACCIONES].mean() * 12
capm_tabla[["beta", "t_beta", "alfa_anual", "t_alfa", "r2", "retorno en exceso"]].round(2).sort_values("beta")

In [ ]:
# La linea caracteristica de una accion: la regresion, en un grafico
T = "SCCO"
datos = pd.concat([ff["Mkt-RF"], exceso[T]], axis=1).dropna() * 100
fila = capm_tabla.loc[T]

fig, ax = plt.subplots(figsize=(7, 4.8))
ax.scatter(datos["Mkt-RF"], datos[T], s=14, color="#000080", alpha=0.6)
x = np.array([datos["Mkt-RF"].min(), datos["Mkt-RF"].max()])
ax.plot(x, fila["alfa_anual"] / 12 * 100 + fila["beta"] * x, color="#E36C0A", lw=2,
        label=f"beta = {fila['beta']:.2f} | R2 = {fila['r2']:.2f}")
ax.axhline(0, color="gray", lw=0.5); ax.axvline(0, color="gray", lw=0.5)
ax.set_xlabel("Exceso de retorno del mercado (% mensual)"); ax.set_ylabel(f"Exceso de retorno de {T} (% mensual)")
ax.set_title(f"Linea caracteristica de {T}"); ax.grid(alpha=0.3); ax.legend(); plt.tight_layout(); plt.show()

**Lectura:** la pendiente es el beta; la dispersión alrededor de la recta es el riesgo idiosincrático. Mira la columna `r2` de la tabla: en la mayoría de acciones el mercado explica menos de la mitad de la varianza, y en las de la BVL casi nada. ¿Significa que las acciones peruanas no tienen riesgo sistemático? No: significa que el mercado de Estados Unidos no es **su** mercado. La elección del índice importa (la crítica de Roll, en versión práctica). Y mira los estadísticos $t$ de los alfas: ¿alguno supera 2?

## 3. La SML con datos: ¿se cumple?

La teoría dice que el retorno en exceso promedio debería ser $\beta$ por la prima del mercado: una recta que pasa por el origen con pendiente igual a la prima. Comparamos esa recta con la que mejor ajusta a nuestros 14 puntos.

In [ ]:
prima_mercado = ff.loc[exceso.index, "Mkt-RF"].mean() * 12
corte = sm.OLS(capm_tabla["retorno en exceso"], sm.add_constant(capm_tabla["beta"])).fit()
print(f"SML teorica:  intercepto 0.0%  | pendiente {prima_mercado:.1%} (la prima del mercado en la muestra)")
print(f"SML empirica: intercepto {corte.params['const']:.1%} | pendiente {corte.params['beta']:.1%} | R2 = {corte.rsquared:.2f}")

fig, ax = plt.subplots(figsize=(8, 5))
x = np.array([0, capm_tabla["beta"].max() * 1.1])
ax.plot(x, prima_mercado * x * 100, color="#000080", lw=2.5, label="SML teorica")
ax.plot(x, (corte.params["const"] + corte.params["beta"] * x) * 100, color="#E36C0A", lw=2, ls="--", label="SML empirica (ajuste a los 14 puntos)")
ax.scatter(capm_tabla["beta"], capm_tabla["retorno en exceso"] * 100, color="black", s=25, zorder=3)
for t in ACCIONES:
    ax.annotate(t.replace(".LM", ""), (capm_tabla.loc[t, "beta"], capm_tabla.loc[t, "retorno en exceso"] * 100),
                xytext=(4, 3), textcoords="offset points", fontsize=8)
ax.set_xlabel("Beta"); ax.set_ylabel("Retorno en exceso anualizado (%)")
ax.set_title("La SML con datos"); ax.grid(alpha=0.3); ax.legend(); plt.tight_layout(); plt.show()

**Preguntas de discusión:** la distancia vertical de cada punto a la SML teórica es su alfa. ¿El intercepto empírico es cero, como pide el CAPM? El resultado clásico de la literatura es una SML empírica con intercepto positivo y más plana que la teórica: las acciones de beta bajo rinden más de lo que el modelo predice. ¿Qué salió aquí?

Y una trampa para pensar: si casi todos nuestros puntos quedaron **sobre** la SML teórica, ¿encontramos 14 alfas positivos? No. Elegimos empresas conocidas que sobrevivieron y crecieron durante la década: es **sesgo de selección** (y de supervivencia). Una muestra armada mirando el pasado siempre le gana al modelo. Con 14 acciones y 10 años esto es una ilustración, no una prueba: las pruebas serias usan miles de acciones, décadas de datos y portafolios ordenados por beta.

## 4. Tres factores: qué carga cada acción

Misma regresión, con SMB y HML además del mercado.

In [ ]:
ff3_tabla = pd.DataFrame({t: regresion_factores(exceso[t], ff[["Mkt-RF", "SMB", "HML"]]) for t in ACCIONES}).T
resumen = pd.DataFrame({
    "b (mercado)": ff3_tabla["b_Mkt-RF"], "s (SMB)": ff3_tabla["b_SMB"], "t_s": ff3_tabla["t_SMB"],
    "h (HML)": ff3_tabla["b_HML"], "t_h": ff3_tabla["t_HML"],
    "alfa CAPM": capm_tabla["alfa_anual"], "alfa FF3": ff3_tabla["alfa_anual"],
    "R2 CAPM": capm_tabla["r2"], "R2 FF3": ff3_tabla["r2"],
})
resumen.round(2).sort_values("h (HML)")

**Lectura:** una carga es significativa, como regla práctica, si su $|t|$ supera 2. Arriba de la tabla están las acciones de crecimiento ($h$ negativo): ¿cuáles son? Abajo, las de valor: ¿tiene sentido que un banco y una petrolera carguen fuerte a HML? Compara `R2 CAPM` con `R2 FF3`: ¿a quién le ayudan más los factores adicionales? Y un recordatorio: estos son factores de Estados Unidos; para las acciones peruanas lo correcto sería usar factores de mercados emergentes (`get_factores("emergentes")`). Queda como ejercicio.

## 5. Análisis de estilo: leer un fondo por sus cargas

Ahora lo mismo con fondos. La gracia: sin abrir el prospecto, las cargas delatan lo que el fondo hace.

In [ ]:
estilo = pd.DataFrame({t: regresion_factores(exceso[t], ff[["Mkt-RF", "SMB", "HML"]]) for t in FONDOS}).T
estilo_capm = pd.DataFrame({t: regresion_factores(exceso[t], ff[["Mkt-RF"]]) for t in FONDOS}).T

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.axhline(0, color="gray", lw=0.8); ax.axvline(0, color="gray", lw=0.8)
ax.scatter(estilo["b_HML"], estilo["b_SMB"], s=70, color="#000080", zorder=3)
for t in FONDOS:
    ax.annotate(t, (estilo.loc[t, "b_HML"], estilo.loc[t, "b_SMB"]),
                xytext=(6, 5), textcoords="offset points", fontsize=9)
ax.set_xlabel("Carga a HML  (crecimiento  <-  0  ->  valor)"); ax.set_ylabel("Carga a SMB  (grandes  <-  0  ->  pequenas)")
ax.set_title("Mapa de estilo segun las cargas de Fama-French"); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

pd.DataFrame({"fondo": pd.Series(FONDOS), "b": estilo["b_Mkt-RF"].round(2), "s": estilo["b_SMB"].round(2),
              "h": estilo["b_HML"].round(2), "R2": estilo["r2"].round(2)})

**Para discutir:** ubica cada fondo en el mapa **antes** de leer su nombre. ¿Dónde caen el Russell 2000 y su versión de valor? ¿Y los dos fondos de Vanguard? ¿Qué dice el mapa de Berkshire Hathaway, el vehículo de Warren Buffett: grandes o pequeñas, valor o crecimiento? Fíjate en los $R^2$: para los fondos diversificados de Estados Unidos pasan de 0.9, porque el riesgo idiosincrático ya se diversificó; tres números describen casi todo lo que el fondo hace. Las excepciones enseñan: Berkshire es una sola empresa y EPU es otro mercado.

## 6. El caso ARKK: un alfa que depende del modelo

ARK Innovation fue el fondo estrella de 2020 y uno de los peores de 2022. ¿Tuvo alfa? Depende de a quién le preguntes.

In [ ]:
F = "ARKK"
print(f"{F} con CAPM: beta = {estilo_capm.loc[F, 'b_Mkt-RF']:.2f} | alfa anual = {estilo_capm.loc[F, 'alfa_anual']:+.1%} (t = {estilo_capm.loc[F, 't_alfa']:.2f})")
print(f"{F} con FF3:  b = {estilo.loc[F, 'b_Mkt-RF']:.2f}, s = {estilo.loc[F, 'b_SMB']:.2f}, h = {estilo.loc[F, 'b_HML']:.2f} | alfa anual = {estilo.loc[F, 'alfa_anual']:+.1%} (t = {estilo.loc[F, 't_alfa']:.2f})")

# Atribucion de retorno: de donde salio el retorno en exceso promedio del fondo
primas = ff.loc[exceso[F].dropna().index, ["Mkt-RF", "SMB", "HML"]].mean() * 12
aporte = pd.Series({"Mercado": estilo.loc[F, "b_Mkt-RF"] * primas["Mkt-RF"], "SMB (tamano)": estilo.loc[F, "b_SMB"] * primas["SMB"],
                    "HML (valor)": estilo.loc[F, "b_HML"] * primas["HML"], "Alfa (seleccion)": estilo.loc[F, "alfa_anual"]})
aporte["TOTAL = retorno en exceso"] = aporte.sum()
print("\nAtribucion del retorno en exceso anualizado:")
print((aporte * 100).round(2).to_string())
print(f"(comprobacion: retorno en exceso observado = {exceso[F].mean() * 12:.2%})")

**Lectura profesional:** en la muestra con que se preparó esta clase (noviembre de 2016 a julio de 2026), el CAPM le asigna al fondo un alfa cercano a $-6\%$ anual y el modelo de tres factores lo deja cerca de cero. ¿Qué cambió? El CAPM solo ve un beta de mercado muy alto; el modelo de tres factores reconoce que el fondo se comporta como una cartera de empresas pequeñas y de crecimiento extremo (mira $s$ y $h$), y que en esa década el factor tamaño pagó negativo. La tabla de atribución lo cuantifica: cada fila es carga por prima, y la suma reproduce el retorno observado. Ninguno de los dos alfas es estadísticamente distinto de cero. Moraleja doble: el alfa depende del modelo, y un alfa sin su estadístico $t$ es una anécdota. (Tus números pueden diferir un poco: la muestra se mueve con la fecha en que ejecutes.)

Esto es exactamente lo que hace un comité cuando evalúa a un gestor: separa lo que ganó por exposiciones que se compran baratas con un ETF de lo que ganó por habilidad. La próxima semana lo formalizamos.

## 7. Cierre

Desde hoy quedan en la librería: `utils/french.py` con `get_factores()` (tres y cinco factores, *momentum* y mercados emergentes) y, en `utils/finanzas.py`, `alfa_capm()`, `descomponer_riesgo()`, `retorno_multifactor()` y `regresion_factores()`.

**Tarea** (`clase09_tarea.ipynb`): el modelo de factores de tu empresa y el análisis de estilo de un fondo que elijas. Entrega hasta el lunes, vía commit.

**Próxima semana:** gestión activa y pasiva, rebalanceo y evaluación de desempeño (Sharpe, Treynor, alfa de Jensen, ratio de información) y valor en riesgo. **PC2: miércoles 11/11**, sobre las semanas 6 a 10.